# Pronto-Socorro: lendo o Data Lake com DuckDB

Os atendimentos enriquecidos pelo ksqlDB foram gravados pelo **Sink Connector** no bucket `bronze` do MinIO, em arquivos JSON.

Neste notebook usamos o **DuckDB** para consultar esses arquivos com SQL, direto no MinIO, sem copiar nada.

## 1. Importando o DuckDB

In [1]:
import duckdb

con = duckdb.connect()

## 2. Conectando no MinIO

A extensão `httpfs` permite ao DuckDB ler arquivos de um S3. A *secret* guarda a mesma credencial usada pelo Sink Connector (`cursolab`).

In [2]:
con.sql("INSTALL httpfs")
con.sql("LOAD httpfs")

con.sql("""
    CREATE OR REPLACE SECRET minio (
        TYPE s3,
        KEY_ID 'cursolab',
        SECRET 'cursolab',
        ENDPOINT 'minio:9000',
        URL_STYLE 'path',
        USE_SSL false,
        REGION 'us-east-1'
    )
""")

┌─────────┐
│ Success │
│ boolean │
├─────────┤
│ true    │
└─────────┘

## 3. Consultando os atendimentos

O Sink grava os arquivos em `bronze/topics/atendimentos-enriquecidos/partition=0/`. O `*` lê todos os arquivos de uma vez.

> Os nomes das colunas vêm em MAIÚSCULAS porque é assim que o ksqlDB escreve o JSON.

In [ ]:
con.sql("""
    SELECT *
    FROM read_json_auto('s3://bronze/topics/atendimentos-enriquecidos/*/*.json')
""").df()

## 4. Um SELECT com filtro

Só os casos mais graves, do mais novo para o mais antigo:

In [ ]:
con.sql("""
    SELECT atendimento_id, paciente, idade, medico, especialidade, classificacao_risco, queixa
    FROM read_json_auto('s3://bronze/topics/atendimentos-enriquecidos/*/*.json')
    WHERE classificacao_risco IN ('VERMELHO', 'LARANJA')
    ORDER BY atendimento_id DESC
""").df()

---
# Camada silver: limpeza + upsert

A bronze guarda **tudo o que chegou**, do jeito que chegou: o mesmo atendimento aparece várias vezes (cada mudança de status é um evento), pode ter texto com espaços sobrando, maiúsculas e minúsculas misturadas, e campos vazios.

A **silver** guarda **uma linha por atendimento**, limpa e na versão mais recente. Para isso fazemos um **upsert**:

| Situação | Ação |
|---|---|
| `atendimento_id` ainda não existe na silver | **INSERT** |
| `atendimento_id` já existe na silver | **UPDATE** |

> Antes de rodar: crie o bucket `silver` no MinIO (http://localhost:9001 → **Buckets** → **Create Bucket**).

## 5. Caminhos da bronze e da silver

In [ ]:
BRONZE = "s3://bronze/topics/atendimentos-enriquecidos/*/*.json"
SILVER = "s3://silver/atendimentos/aendimentos.parquet"

## 6. Limpeza dos dados da bronze

| Regra | Como |
|---|---|
| Esquema fixo (tipos definidos) | `read_json(..., columns = {...})` |
| Data/hora no fuso de São Paulo | `CAST(... AS TIMESTAMPTZ) AT TIME ZONE 'America/Sao_Paulo'` |
| Tirar espaços sobrando | `TRIM(...)` |
| Padronizar códigos em maiúsculas | `UPPER(...)` em convênio, classificação e status |
| Queixa vazia vira `NAO INFORMADA` | `COALESCE(NULLIF(TRIM(queixa), ''), ...)` |
| Descartar registros sem paciente ou médico | `WHERE paciente IS NOT NULL AND medico IS NOT NULL` |
| Ficar só com a **última versão** de cada atendimento | `QUALIFY row_number() ... = 1` |

> **Como saber qual é a última versão?** O nome de cada arquivo tem o **offset** do Kafka (`atendimentos-enriquecidos+0+0000000003.json`), e dentro do arquivo as linhas estão em ordem. Arquivo mais "novo" + última linha = versão mais recente.

In [ ]:
con.sql(f"""
    CREATE OR REPLACE VIEW bronze_limpo AS
    WITH bronze AS (
        SELECT *, row_number() OVER () AS linha
        FROM read_json('{BRONZE}', filename = true, columns = {{
            'ATENDIMENTO_ID': 'INTEGER', 'ABERTO_EM': 'VARCHAR',
            'PACIENTE_ID': 'INTEGER', 'PACIENTE': 'VARCHAR', 'IDADE': 'INTEGER',
            'CONVENIO': 'VARCHAR', 'MEDICO_ID': 'INTEGER', 'MEDICO': 'VARCHAR',
            'ESPECIALIDADE': 'VARCHAR', 'CLASSIFICACAO_RISCO': 'VARCHAR',
            'QUEIXA': 'VARCHAR', 'STATUS': 'VARCHAR', 'OPERACAO': 'VARCHAR'
        }})
    )
    SELECT
        atendimento_id                                                  AS atendimento_id,
        CAST(aberto_em AS TIMESTAMPTZ) AT TIME ZONE 'America/Sao_Paulo'  AS aberto_em,
        paciente_id                                                     AS paciente_id,
        TRIM(paciente)                                                  AS paciente,
        idade                                                           AS idade,
        UPPER(TRIM(convenio))                                           AS convenio,
        medico_id                                                       AS medico_id,
        TRIM(medico)                                                    AS medico,
        TRIM(especialidade)                                             AS especialidade,
        UPPER(TRIM(classificacao_risco))                                AS classificacao_risco,
        COALESCE(NULLIF(TRIM(queixa), ''), 'NAO INFORMADA')             AS queixa,
        UPPER(TRIM(status))                                             AS status,
        now() AT TIME ZONE 'America/Sao_Paulo'                          AS atualizado_em
    FROM bronze
    WHERE paciente IS NOT NULL
      AND medico   IS NOT NULL
    QUALIFY row_number() OVER (
        PARTITION BY atendimento_id
        ORDER BY filename DESC, linha DESC
    ) = 1
""")

con.sql("SELECT * FROM bronze_limpo ORDER BY atendimento_id").df()

## 7. Upsert na silver

O DuckDB não altera um arquivo Parquet "no lugar". Então o fluxo é:

1. Cria uma tabela com **chave primária** (`atendimento_id`)
2. Carrega nela o que já está na silver
3. Faz o **upsert** com `INSERT ... ON CONFLICT (atendimento_id) DO UPDATE`
4. Grava a tabela de volta no MinIO, em **Parquet**

Rode esta célula mais de uma vez: na segunda execução, os atendimentos que já existem são **atualizados**, não duplicados.

In [13]:
# 1. Tabela da silver com chave primária: é ela que permite o upsert
con.sql("""
    CREATE OR REPLACE TABLE silver_atendimentos (
        atendimento_id      INTEGER PRIMARY KEY,
        aberto_em           TIMESTAMP,
        paciente_id         INTEGER,
        paciente            VARCHAR,
        idade               INTEGER,
        convenio            VARCHAR,
        medico_id           INTEGER,
        medico              VARCHAR,
        especialidade       VARCHAR,
        classificacao_risco VARCHAR,
        queixa              VARCHAR,
        status              VARCHAR,
        atualizado_em       TIMESTAMP
    )
""")

# 2. Carrega o que já está na silver (na primeira execução o arquivo ainda não existe)
try:
    con.sql(f"INSERT INTO silver_atendimentos SELECT * FROM read_parquet('{SILVER}')")
    print("Silver atual:", con.sql("SELECT count(*) FROM silver_atendimentos").fetchone()[0], "atendimentos")
except duckdb.Error:
    print("Primeira carga: a silver ainda está vazia")

novos = con.sql("""
    SELECT count(*) FROM bronze_limpo
    WHERE atendimento_id NOT IN (SELECT atendimento_id FROM silver_atendimentos)
""").fetchone()[0]

# 3. UPSERT: insere os atendimentos novos e atualiza os que já existem
con.sql("""
    INSERT INTO silver_atendimentos
    SELECT * FROM bronze_limpo
    ON CONFLICT (atendimento_id) DO UPDATE SET
        aberto_em           = EXCLUDED.aberto_em,
        paciente_id         = EXCLUDED.paciente_id,
        paciente            = EXCLUDED.paciente,
        idade               = EXCLUDED.idade,
        convenio            = EXCLUDED.convenio,
        medico_id           = EXCLUDED.medico_id,
        medico              = EXCLUDED.medico,
        especialidade       = EXCLUDED.especialidade,
        classificacao_risco = EXCLUDED.classificacao_risco,
        queixa              = EXCLUDED.queixa,
        status              = EXCLUDED.status,
        atualizado_em       = EXCLUDED.atualizado_em
""")

total_bronze = con.sql("SELECT count(*) FROM bronze_limpo").fetchone()[0]
print(f"Inseridos: {novos} | Atualizados: {total_bronze - novos}")

# 4. Grava a silver de volta no MinIO, em Parquet
con.sql(f"COPY silver_atendimentos TO '{SILVER}' (FORMAT parquet)")
print("Silver gravada em", SILVER)

Primeira carga: a silver ainda está vazia
Inseridos: 138 | Atualizados: 0
Silver gravada em s3://silver/atendimentos/aendimentos.parquet


## 8. Conferindo a silver

Uma linha por atendimento, dados padronizados e no formato Parquet (colunar, com tipos e comprimido).

In [ ]:
con.sql(f"""
    SELECT atendimento_id, aberto_em, paciente, idade, convenio, medico,
           classificacao_risco, queixa, status, atualizado_em
    FROM read_parquet('{SILVER}')
    ORDER BY atendimento_id
""").df()